### 1. Zhu et al , 2018 : On the radio detectability of circumplanetary discs





In [ ]:
import numpy as np
from astropy.constants import G, M_jup, R_jup ,stefan_boltzmann

In [ ]:
# define the constants needed in appropriate cgs units
G = G.to('cm^3 / (g s^2)')
M_jup = M_jup.to('g')
R_jup = R_jup.to('cm')
sigmaSB = stefan_boltzmann.to('erg / (cm^2 s K^4)')

In [ ]:
# Where to store the variables is a problem especially if I am exploring a parameter space 




def surface_density(alpha ,Rkappa):
    """alpha: viscosity constant
       Rkappa: Rosseland mean opacity (cm^2/g)
    """

    return Sigma




def mid_plane_temperature_profile(R, planet_irr):
    """
    R: Radius array (cm)
    """
    T_ISM = 10  # K , as the value Zhu took

    if planet_irr == True:
        L_irr = # Marley 2007, Spiegel & Burrows 2012
    else:
        L_irr = G*M_p*Mdot_p/(2*R_p) # from the boundary layer


    T_irr = 100 * (R / (1 * R_jup))**(-3/4)  # K

    T_ext = (T_ISM**4+T_irr**4)**(1/4)
    T_c = (T_ext**4 + (9 * G * Mp * Mdot_p * Sigma * kappaR) / (128 * np.pi * sigmaSB * R**3) * (1 - np.sqrt(R_in / R)))**(1/4)
    return T_c



def brightness_temperature(tau_mm):
    if tau_mm > 0.5:
        T_b = (3/8)*


In [ ]:
# --- CPD Section 2 equations in Python -----------------------
# Source: Zhu, Andrews & Isella (2018) MNRAS 479, 1850 (arXiv:1708.07287)
# -------------------------------------------------------------

import numpy as np

# Physical constants (cgs)
G      = 6.67430e-8               # cm^3 g^-1 s^-2
sigmaB = 5.670374419e-5           # erg cm^-2 s^-1 K^-4 (Stefan–Boltzmann)
kB     = 1.380649e-16             # erg K^-1
mH     = 1.6735575e-24            # g

# Helper: Keplerian angular frequency around the planet
def omega_k(Mp, R):
    """Keplerian angular frequency Ω = sqrt(G Mp / R^3) [s^-1]."""
    return np.sqrt(G * Mp / R**3)

# Eq. (1): viscous disc effective temperature (per radius)
def Teff_visc(Mp, Mdot_p, R, Rin):
    """
    T_eff^4 = 3 G Mp Mdot_p / (8 π σ R^3) * [1 - (Rin/R)^{1/2}]
    Returns T_eff [K].
    """
    term = (3 * G * Mp * Mdot_p) / (8 * np.pi * sigmaB * R**3)
    f = (1.0 - np.sqrt(Rin / R))
    return (term * f)**0.25

# Eq. (3): irradiation temperature from a bright planet or boundary layer
def Tirr(Lirr, R):
    """
    T_irr = (L_irr / (4 π σ R^2))^{1/4}
    L_irr can be the planet luminosity or 0.5 * L_acc for a boundary layer.
    """
    return (Lirr / (4 * np.pi * sigmaB * R**2))**0.25

# External/background temperature (text under Eq. 2)
def Text(T_ISM=10.0, T_irr=None):
    """
    T_ext^4 = T_ISM^4 + T_irr^4  (omit T_irr term if not provided)
    """
    if T_irr is None:
        return float(T_ISM)
    return (T_ISM**4 + T_irr**4)**0.25

# Eq. (2): vertical temperature profile (grey atmosphere)
def T_at_tau(Teff, tau_R, T_ext):
    """
    T(τ)^4 = (3/8) τ_R T_eff^4 + T_ext^4
    Returns T(τ_R) [K].
    """
    return ((3/8) * tau_R * Teff**4 + T_ext**4)**0.25

# Midplane Rosseland optical depth and midplane temperature (Eqs. 2 & 4)
def tau_R_midplane(Sigma, kappa_R):
    """τ_R(midplane) ≈ (1/2) κ_R Σ   (Σ is two-sided surface density)."""
    return 0.5 * kappa_R * Sigma

def Tc_midplane_from_Sigma(Mp, Mdot_p, Sigma, kappa_R, R, Rin, T_ext):
    """
    Eq. (4): T_c^4 = 9 G Mp Mdot_p Σ κ_R / (128 π σ R^3) [1 - (Rin/R)^{1/2}] + T_ext^4
    Returns T_c [K].
    """
    A = (9 * G * Mp * Mdot_p * Sigma * kappa_R) / (128 * np.pi * sigmaB * R**3)
    f = (1.0 - np.sqrt(Rin / R))
    return (A * f + T_ext**4)**0.25

# Eq. (5): steady α-disc mass transport relation ν Σ = Mdot_p / (3π) [1 - (Rin/R)^{1/2}]
def Sigma_from_Tc_alpha(Mp, Mdot_p, alpha, Tc, R, Rin, mu=2.4):
    """
    Using ν = α c_s^2 / Ω with c_s^2 = k_B T_c / (μ m_H), Ω=Ω_K(Mp,R).
    Returns Σ [g cm^-2].
    """
    cs2 = kB * Tc / (mu * mH)
    nu  = alpha * cs2 / omega_k(Mp, R)
    return (Mdot_p / (3*np.pi) * (1.0 - np.sqrt(Rin/R))) / nu

# Solve the coupled system {Eq. (4), Eq. (5)} for (Σ, T_c)
def solve_sigma_Tc(Mp, Mdot_p, alpha, kappa_R, R, Rin, T_ext, mu=2.4, 
                   tol=1e-6, max_iter=200):
    """
    Fixed-point / Newton-like iteration:
      1) guess Σ, compute T_c from Eq. (4)
      2) update Σ from Eq. (5) using ν(α, T_c)
    Converges quickly for typical CPD parameters.
    """
    # Initial guess: optically thick viscous disc rough scaling
    Sigma = 10.0  # g cm^-2 (seed; method is robust for wide range)
    for _ in range(max_iter):
        Tc_old   = Tc_midplane_from_Sigma(Mp, Mdot_p, Sigma, kappa_R, R, Rin, T_ext)
        Sigma_new = Sigma_from_Tc_alpha(Mp, Mdot_p, alpha, Tc_old, R, Rin, mu=mu)
        if np.abs(Sigma_new - Sigma) / (Sigma + 1e-30) < tol:
            return Sigma_new, Tc_old
        Sigma = 0.5*Sigma + 0.5*Sigma_new  # under-relaxation for stability
    # If not converged, still return the last iterate
    return Sigma, Tc_midplane_from_Sigma(Mp, Mdot_p, Sigma, kappa_R, R, Rin, T_ext)

# (Optional) Brightness temperature at a given (mm/cm) opacity (Eq. 8; piecewise)
def Tb_mm(Sigma, Tc, T_ext, kappa_mm):
    """
    τ_mm = 0.5 κ_mm Σ.  If optically thick at the observing λ, use the grey-slab
    brightness at τ≈1; if thin, use Rayleigh-Jeans optically-thin limit.
    This mirrors the piecewise prescription summarized under Eq. (8).
    """
    tau_mm = 0.5 * kappa_mm * Sigma
    if tau_mm > 0.5:
        # approximate temperature at τ≈1 in a grey atmosphere:
        # T(τ=1)^4 ≈ (3/8)*(1)*T_c^4*(κ_R/κ_mm) + T_ext^4    (scales used in the paper)
        return ((3/8) * (Tc**4) + T_ext**4)**0.25
    else:
        # optically thin RJ limit: Tb ≈ 2 τ_mm T_c
        return 2.0 * tau_mm * Tc

# Eq. (9): disc-averaged brightness temperature over the CPD surface
def Tb_avg_over_disc(Rin, Rout, Tb_of_R):
    """
    T̄_b = ( ∫_{Rin}^{Rout} Tb(R) * 2πR dR ) / (π (Rout^2 - Rin^2))
    Pass Tb_of_R as a callable returning Tb at radius R.
    """
    # simple log-space quadrature that behaves well near Rin
    Rs = np.geomspace(Rin, Rout, 256)
    Tb_vals = Tb_of_R(Rs)
    num = np.trapz(Tb_vals * 2*np.pi*Rs, Rs)
    den = np.pi * (Rout**2 - Rin**2)
    return num / den

In [ ]:
# Recreate table 1 from Zhu to check the results